🎯 Goal of Notebook 04

Convert raw telemetry into features that help the model detect:

Abnormal machine behavior

before failures happen.

In [19]:
import numpy as numpy
import pandas as pd

In [20]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"

print(DATA_DIR)

f:\Projects\Advanced-Predictive-Maintenance\data\raw


In [21]:
import pandas as pd

errors = pd.read_csv(DATA_DIR / "PdM_errors.csv")
telemetry = pd.read_csv(DATA_DIR / "PdM_telemetry.csv")
machines = pd.read_csv(DATA_DIR / "PdM_machines.csv")
maint = pd.read_csv(DATA_DIR / "PdM_maint.csv")
failures = pd.read_csv(DATA_DIR / "PdM_failures.csv")

In [22]:
telemetry["datetime"] = pd.to_datetime(telemetry["datetime"])

master_df = telemetry.merge(
    machines,
    on="machineID",
    how="left"
)

master_df = master_df.sort_values(
    ["machineID", "datetime"]
)

master_df.head()

,datetime,machineID,volt,rotate,pressure,vibration,model,age
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,model3,18
1,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,model3,18
2,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,model3,18
3,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,model3,18
4,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,model3,18


Step 2: Rolling Features
Rolling Mean (24 Hours)

For vibration:

In [23]:
master_df["vibration_mean_24h"] = (
    master_df.groupby("machineID")["vibration"]
    .transform(
        lambda x: x.rolling(
            window=24,
            min_periods=1
        ).mean()
    )
)

In [24]:
sensors = [
    "volt",
    "rotate",
    "pressure",
    "vibration"
]

for col in sensors:

    master_df[f"{col}_mean_24h"] = (
        master_df.groupby("machineID")[col]
        .transform(
            lambda x: x.rolling(
                24,
                min_periods=1
            ).mean()
        )
    )

Rolling Standard Deviation

In [25]:
for col in sensors:

    master_df[f"{col}_std_24h"] = (
        master_df.groupby("machineID")[col]
        .transform(
            lambda x: x.rolling(
                24,
                min_periods=1
            ).std()
        )
    )

1 Hour Lag

In [26]:
for col in sensors:

    master_df[f"{col}_lag1"] = (
        master_df.groupby("machineID")[col]
        .shift(1)
    )

6 Hour Lag

In [27]:
for col in sensors:

    master_df[f"{col}_lag6"] = (
        master_df.groupby("machineID")[col]
        .shift(6)
    )

24 Hour Lag

In [28]:
for col in sensors:

    master_df[f"{col}_lag24"] = (
        master_df.groupby("machineID")[col]
        .shift(24)
    )

Step 5: Trend Features
Why?

We found rotation decreases before failures.

Trend captures:

Current - Previous

Rate of Change

In [29]:
for col in sensors:

    master_df[f"{col}_change"] = (
        master_df[col]
        -
        master_df[f"{col}_lag1"]
    )

Percentage Change

In [30]:
for col in sensors:

    master_df[f"{col}_pct_change"] = (
        master_df.groupby("machineID")[col]
        .pct_change()
    )

Maintenance Features

In [31]:
maint.head()

,datetime,machineID,comp
0,2014-06-01 06:00:00,1,comp2
1,2014-07-16 06:00:00,1,comp4
2,2014-07-31 06:00:00,1,comp3
3,2014-12-13 06:00:00,1,comp1
4,2015-01-05 06:00:00,1,comp4


In [32]:
maint["datetime"] = pd.to_datetime(
    maint["datetime"]
)

maint = maint.sort_values(
    ["machineID","datetime"]
)

In [33]:
master_df = pd.merge_asof(
    master_df.sort_values("datetime"),
    maint.sort_values("datetime"),
    on="datetime",
    by="machineID",
    direction="backward"
)

Rename:

In [34]:
maint_feature = maint.copy()

In [35]:
maint_feature = maint_feature.rename(
    columns={
        "datetime": "maintenance_time"
    }
)

In [36]:
maint_feature = maint_feature.sort_values(
    ["machineID", "maintenance_time"]
)

In [37]:
master_df = master_df.sort_values(
    ["machineID", "datetime"]
)

This is a powerful feature.

In [39]:
master_df.shape

(876100, 37)

In [40]:
master_df.head()

,datetime,machineID,volt,rotate,pressure,vibration,model,age,vibration_mean_24h,volt_mean_24h,...,vibration_lag24,volt_change,rotate_change,pressure_change,vibration_change,volt_pct_change,rotate_pct_change,pressure_pct_change,vibration_pct_change,comp
0,2015-01-01 06:00:00,1,176.217853,418.504078,113.077935,45.087686,model3,18,45.087686,176.217853,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,comp1
168,2015-01-01 07:00:00,1,162.879223,402.747490,95.460525,43.413973,model3,18,44.250829,169.548538,...,NaN,-13.338630,-15.756589,-17.617410,-1.673713,-0.075694,-0.037650,-0.155799,-0.037121,comp1
241,2015-01-01 08:00:00,1,170.989902,527.349825,75.237905,34.178847,model3,18,40.893502,170.028993,...,NaN,8.110680,124.602336,-20.222621,-9.235126,0.049796,0.309381,-0.211843,-0.212722,comp1
398,2015-01-01 09:00:00,1,162.462833,346.149335,109.248561,41.122144,model3,18,40.950662,168.137453,...,NaN,-8.527069,-181.200490,34.010656,6.943297,-0.049869,-0.343606,0.452042,0.203146,comp1
419,2015-01-01 10:00:00,1,157.610021,435.376873,111.886648,25.990511,model3,18,37.958632,166.031967,...,NaN,-4.852812,89.227538,2.638087,-15.131633,-0.029870,0.257772,0.024148,-0.367968,comp1


Droping the component Column

In [41]:
master_df = master_df.drop(
    columns=["comp"],
    errors="ignore"
)

Drop Data Column because They don't capture patterns

In [42]:
master_df = master_df.drop(
    columns=["datetime"]
)

Handle model

In [43]:
master_df = pd.get_dummies(
    master_df,
    columns=["model"],
    drop_first=True
)

In [ ]:
master_df.head()

Handle NAN's

In [ ]:
master_df.isnull().sum().sort_values(
    ascending=False
).head(20)

In [46]:
master_df = master_df.fillna(0)

Check Infinite Values

In [ ]:
import numpy as np

np.isinf(master_df.select_dtypes(
    include=np.number
)).sum()

Before Training Run These Checks
Missing Values , 
Infinite Values ,
Data Types


In [49]:
print(master_df.isnull().sum().sum())
print(np.isinf(master_df.select_dtypes(include=np.number)).sum().sum())
print(master_df.dtypes.value_counts())

0
0
float64    32
bool        3
int64       2
Name: count, dtype: int64


In [50]:
feature_df = master_df.select_dtypes(
    include=["int64", "float64", "bool"]
)

In [51]:
feature_df.shape

(876100, 37)